<a href="https://colab.research.google.com/github/Komal5578/DWM/blob/main/PageRank_exp10_DWM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Graph Used
The graph has 4 nodes with these links:

| Node | Links to |
|------|----------|
| 1 | 2, 3, 4 |
| 2 | 3, 4 |
| 3 | 1 |
| 4 | 1, 3 |

The matrix **A** is built so that column j holds 1/(out-degree of node j) for every page that j links to.

In [ ]:
import numpy as np
import networkx as nx

In [ ]:
A = np.array([[0,   0,   1, 0.5],
              [1/3, 0,   0, 0  ],
              [1/3, 0.5, 0, 0.5],
              [1/3, 0.5, 0, 0  ]])
n = A.shape[0]
print("Matrix A:\n", A)
print("Column sums:", A.sum(axis=0))  # each should be 1

Matrix A:
 [[0.         0.         1.         0.5       ]
 [0.33333333 0.         0.         0.        ]
 [0.33333333 0.5        0.         0.5       ]
 [0.33333333 0.5        0.         0.        ]]
Column sums: [1. 1. 1. 1.]


In [ ]:
def pagerank_simple(A, max_iter=100, tol=1e-8):
    n = A.shape[0]
    v = np.ones(n) / n
    print("V0 =", v)
    for i in range(1, max_iter + 1):
        v_new = A @ v
        if np.allclose(v_new, v, atol=tol):
            return v_new, i
        v = v_new
    return v, max_iter

v, k = pagerank_simple(A)
print("PageRank vector:", v)
print("Iterations:", k)

V0 = [0.25 0.25 0.25 0.25]
PageRank vector: [0.38709741 0.12903205 0.29032246 0.19354808]
Iterations: 21


In [ ]:
def pagerank_damped(A, d=0.85, max_iter=100, tol=1e-8):
    n = A.shape[0]
    v = np.ones(n) / n
    for i in range(1, max_iter + 1):
        v_new = (1 - d) / n + d * (A @ v)
        if np.abs(v_new - v).sum() < tol:
            return v_new, i
        v = v_new
    return v, max_iter

v2, k2 = pagerank_damped(A, d=0.85)
print("Damped PageRank:", v2)
print("Iterations:", k2)

Damped PageRank: [0.36815068 0.14180936 0.28796163 0.20207834]
Iterations: 24


In [ ]:
G = nx.DiGraph([(1,2),(1,3),(1,4),(2,3),(2,4),(3,1),(4,1),(4,3)])
print("networkx PageRank:", nx.pagerank(G, alpha=0.85))
print("Our PageRank     :", dict(zip([1,2,3,4], v2.round(6))))

networkx PageRank: {1: 0.3681509531104541, 2: 0.14180962694364857, 3: 0.28796129337481535, 4: 0.20207812657108176}
Our PageRank     : {1: np.float64(0.368151), 2: np.float64(0.141809), 3: np.float64(0.287962), 4: np.float64(0.202078)}


In [ ]:
L = (A > 0).T.astype(float)

hub  = np.ones(n)
auth = np.ones(n)
for i in range(100):
    new_auth = L.T @ hub
    new_hub  = L @ new_auth
    new_auth /= np.linalg.norm(new_auth)
    new_hub  /= np.linalg.norm(new_hub)
    if np.allclose(new_auth, auth, atol=1e-8) and np.allclose(new_hub, hub, atol=1e-8):
        break
    auth, hub = new_auth, new_hub

print("Authority scores:", auth)
print("Hub scores:", hub)
print("Iterations:", i + 1)

Authority scores: [0.22943891 0.30627592 0.73941692 0.55390926]
Hub scores: [0.69994292 0.5659248  0.10039631 0.42394529]
Iterations: 15


In [ ]:
h, a = nx.hits(G)
print("Hubs:", h)
print("Authorities:", a)

Hubs: {1: 0.39098432423031937, 2: 0.31612245554460533, 3: 0.056080340460184584, 4: 0.23681287976489074}
Authorities: {1: 0.1254412278344748, 2: 0.16745199206817146, 3: 0.40426487168209674, 4: 0.30284190841525704}
